# CLIP 기반 Scene Frame 추출 노트북

현재 최종 인스타그램 overlay 파이프라인과 같은 CLIP embedding distance 방식으로 scene을 나누고, 각 scene의 대표 프레임만 추출하는 노트북입니다.

- 모델 예측 없음
- overlay 영상 생성 없음
- 라벨링 후보 이미지 추출 전용
- 결과 저장 위치: `outputs_clip_frame_extraction/`


In [1]:
# 필요한 라이브러리와 기존 파이프라인 함수를 불러옵니다.
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import shotguide_batch_video_inference as base
import shotguide_instagram_overlay_pipeline as pipe

# 노트북 실행 위치를 프로젝트 루트로 사용합니다.
ROOT = Path.cwd()

# 새로 프레임을 추출할 인스타그램 Reels 링크를 여기에 추가합니다.
# 첫 번째 값은 기존 데이터셋처럼 영상 고유 번호(item_id)로 사용됩니다.
EXTRACTION_ITEMS = [
    ("0131", "https://www.instagram.com/reel/DYjePvvTeSB/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA=="),
    ("0132", "https://www.instagram.com/reel/DV7sM11j9mk/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA=="),
    ("0133", "https://www.instagram.com/reel/DYvmCaHpVlW/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA=="),
    ("0134", "https://www.instagram.com/reel/DYcNmJLxAeM/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA=="),
    ("0135", "https://www.instagram.com/reel/DWlU1QYEvCY/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA=="),
    ("0136", "https://www.instagram.com/reel/DW3r7dREtKG/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA=="),
    ("0137", "https://www.instagram.com/reel/DVYFPw3kkTT/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA=="),
    ("0138", "https://www.instagram.com/reel/DYvyGwmpf6c/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA=="),
    ("0139", "https://www.instagram.com/reel/DYq9rB4xxA4/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA=="),
    ("0140", "https://www.instagram.com/reel/DYblPOKhVtF/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA=="),
    ("0141", "https://www.instagram.com/reel/DYEZWBhAux0/?igsh=MTRwa3BwYjE2MGtlOQ=="),
    ("0142", "https://www.instagram.com/reel/DYj3UqZJ741/?igsh=eXI3cXh1NnUzZTRk"),
    ("0143", "https://www.instagram.com/reel/DYj2QYVJLeF/?igsh=MXMybGRmZHdrY2F2cQ=="),
    ("0144", "https://www.instagram.com/reel/DYHM5APzWG1/?igsh=cDNjMmpheWZjczJ1"),
    ("0145", "https://www.instagram.com/reel/DYg_LntBWbv/?igsh=NTR3N21nNzZ0d2Vu"),
    ("0146", "https://www.instagram.com/p/DYl6rvrz67t/"),
    ("0147", "https://www.instagram.com/p/DYUxkrHz4Db/"),
    ("0148", "https://www.instagram.com/p/DYdpaPWhW5g/"),
    ("0149", "https://www.instagram.com/p/DXqwiXwkSAu/"),
    ("0150", "https://www.instagram.com/p/DYd3U0ApeKm/"),
    ("0151", "https://www.instagram.com/p/DVtLk2rk3oi/"),
    ("0152", "https://www.instagram.com/p/DXBvUdcAeHO/"),
    ("0153", "https://www.instagram.com/p/DXQgKSzkz_-/"),
    ("0154", "https://www.instagram.com/p/DYo6sdFTTWT/"),
    ("0155", "https://www.instagram.com/p/DWu6QXOkZQf/"),
    ("0156", "https://www.instagram.com/p/DWvAHNKE9ac/"),
    ("0157", "https://www.instagram.com/p/DV3Qt1wkZXl/"),
    ("0158", "https://www.instagram.com/p/DW8v5bEDzP0/"),
    ("0159", "https://www.instagram.com/p/DX6eSwUh-x9/"),
    ("0160", "https://www.instagram.com/p/DYl_PI0psTg/"),
    ("0161", "https://www.instagram.com/p/DYuJg7Dz-t2/"),
    ("0162", "https://www.instagram.com/p/DYYTvkwJQIm/"),
    ("0163", "https://www.instagram.com/p/DVn7iKJE3li/"),
    ("0164", "https://www.instagram.com/p/DXJvfjSEV7N/"),
    ("0165", "https://www.instagram.com/p/DVS-GW3k-3l/"),
    ("0166", "https://www.instagram.com/p/DVf5ZfaE0-b/"),
    ("0167", "https://www.instagram.com/p/DVn9cumk72M/"),
    ("0168", "https://www.instagram.com/p/DWF6bbeDdSK/"),
    ("0169", "https://www.instagram.com/p/DXNop8lDxfj/"),
    ("0170", "https://www.instagram.com/p/DWX1gloEoXv/"),
    ("0171", "https://www.instagram.com/p/DYY8Zd6p8Z7/"),
    ("0172", "https://www.instagram.com/p/DYQvsu8Bz0V/"),
    ("0173", "https://www.instagram.com/p/DVYsiumDVZY/"),
    ("0174", "https://www.instagram.com/p/DYJ2m1XSfxC/"),
    ("0175", "https://www.instagram.com/p/DVf3N_lE5fA/"),
    ("0176", "https://www.instagram.com/reels/DVfxFZbEnqq/"),
    ("0177", "https://www.instagram.com/reels/DWBQiNjiVqu/"),
    ("0178", "https://www.instagram.com/reels/DWtX5ylkgDB/"),
    ("0179", "https://www.instagram.com/reels/DXHkuIyE4vn/"),
    ("0180", "https://www.instagram.com/reels/DXHAd23komj/"),
    ("0181", "https://www.instagram.com/p/DYBn6CvgBRz/?igsh=aGt4cmp3NTZwNHhk"),
    ("0182", "https://www.instagram.com/p/DXJK7ySDHJ8/?igsh=ZWp5anQ5Zjhrdjh1"),
    ("0183", "https://www.instagram.com/reel/DXvJ1LASEQt/?igsh=MTltYWR1ejE3YWVyaw=="),
    ("0184", "https://www.instagram.com/reel/DVyJpR_iXuX/?igsh=MXJ6eW9ranlteWF1cQ=="),
    ("0185", "https://www.instagram.com/reel/DWam0UAinUp/?igsh=MTh2Z2prdHlucW85NA=="),
    ("0186", "https://www.instagram.com/reel/DW6bbuWjkJT/?igsh=MXA4emsxbjlubWk2"),
    ("0187", "https://www.instagram.com/reel/DV8SfjQgSXD/?igsh=emgyM3A0M2piM3Fv"),
    ("0188", "https://www.instagram.com/reel/DYUFHbrobt0/?igsh=c2swdDA0NTdyeTBq"),
    ("0189", "https://www.instagram.com/reel/DXmAXq_CIaV/?igsh=N25nY3lmc2FvaTV5"),
    ("0190", "https://www.instagram.com/reel/DYfRNHLvqpN/?igsh=MWNmdDRvd2MyaTM4cw=="),
    ("0191", "https://www.instagram.com/reel/DXoSrWrET6J/?igsh=MTI4aTd5YmtlZjczag=="),
    ("0192", "https://www.instagram.com/reel/DYB-OYRyLaO/?igsh=MWdocjZ6dXNiMWVwaA=="),
    ("0193", "https://www.instagram.com/reel/DYossBrSaf-/?igsh=eXk3amg4eGl3NnVx"),
    ("0194", "https://www.instagram.com/reel/DYt86KEStVK/?igsh=dTgzYjd0M2k0MWg1"),
    ("0195", "https://www.instagram.com/reel/DW_UYoBviXv/?igsh=Nng5YzA2Z2l0YTht"),
    ("0196", "https://www.instagram.com/reel/DXgtgqJkSSK/?igsh=MWNwcW5ra3dlZ3phYg=="),
]

# 추출 결과 저장 폴더입니다.
OUTPUT_ROOT = ROOT / "outputs_clip_frame_extraction"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# scene 하나당 추출할 대표 프레임 수입니다.
NUM_FRAME_SAMPLES = 1

base.DEVICE


c:\Users\eunpa\anaconda3\envs\sy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cpu')

## 추출 함수

링크 하나에 대해 다음 작업만 수행합니다.

1. 인스타그램 영상 다운로드
2. CLIP embedding distance 기반 scene detection
3. scene별 대표 프레임 추출
4. 기존 데이터셋과 같은 파일명으로 이미지 저장
5. 메타데이터 CSV 저장


In [2]:
def save_scene_frames_with_dataset_names(video_path: Path, scene_df: pd.DataFrame, output_dir: Path, item_id: str, num_samples=3):
    # 기존 데이터셋과 같은 형식으로 대표 프레임을 저장합니다.
    # 예: 0131_cut_001.jpg, 0131_cut_002.jpg
    output_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    rows = []
    cut_count = 1

    for _, scene in scene_df.iterrows():
        frame_indices = base.sample_frame_indices(
            int(scene.start_frame),
            int(scene.end_frame),
            num_samples=num_samples,
        )
        frame_paths = []

        for frame_idx in frame_indices:
            frame = base.read_frame_at(cap, frame_idx)
            if frame is None:
                continue

            frame_path = output_dir / f"{item_id}_cut_{cut_count:03d}.jpg"
            cv2.imwrite(str(frame_path), frame)
            frame_paths.append(str(frame_path.resolve()))
            cut_count += 1

        row = scene.to_dict()
        row["sampled_frame_indices"] = json.dumps(frame_indices)
        row["sampled_frame_paths"] = json.dumps(frame_paths, ensure_ascii=False)
        rows.append(row)

    cap.release()
    return pd.DataFrame(rows)


def extract_scene_frames_only(item_id: str, url: str, download_index: int, clip_model, clip_preprocess):
    # 기존 파이프라인과 같은 다운로드 함수를 사용합니다.
    video_path = pipe.download_instagram_video(url, download_index)
    shortcode = pipe.get_shortcode(url)

    # 기존 코드처럼 item_id_shortcode 형식의 결과 폴더를 만듭니다.
    video_folder_name = f"{item_id}_{shortcode}"
    video_output_dir = OUTPUT_ROOT / video_folder_name
    frames_dir = video_output_dir / "scene_frames"
    video_output_dir.mkdir(parents=True, exist_ok=True)

    # 현재 최종 파이프라인과 같은 CLIP distance 방식으로 scene을 나눕니다.
    scene_df, distance_df, threshold, info = pipe.detect_scenes_clip_distance(
        video_path,
        clip_model,
        clip_preprocess,
    )

    # scene detection 과정에서 계산된 프레임 간 CLIP 거리 기록입니다.
    distance_df.to_csv(
        video_output_dir / "clip_distance_profile.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # 기존 데이터셋과 같은 이미지 파일명으로 scene 대표 프레임을 저장합니다.
    scene_df = save_scene_frames_with_dataset_names(
        video_path,
        scene_df,
        frames_dir,
        item_id=item_id,
        num_samples=NUM_FRAME_SAMPLES,
    )

    # 라벨링할 때 참고할 scene 메타데이터입니다.
    scene_df.to_csv(
        video_output_dir / "scene_metadata.csv",
        index=False,
        encoding="utf-8-sig",
    )

    return {
        "item_id": item_id,
        "url": url,
        "shortcode": shortcode,
        "video_path": str(video_path.resolve()),
        "output_dir": str(video_output_dir.resolve()),
        "frames_dir": str(frames_dir.resolve()),
        "scene_count": int(len(scene_df)),
        "duration_sec": float(info["duration_sec"]),
        "threshold": float(threshold),
        "num_extracted_frames": int(
            scene_df["sampled_frame_paths"].apply(lambda x: len(json.loads(x))).sum()
        ),
    }


## 실행

`EXTRACTION_ITEMS`에 입력한 `(영상 번호, 링크)`를 순서대로 처리합니다. 결과 이미지는 각 영상 폴더의 `scene_frames/` 아래에 저장됩니다.


In [3]:
# scene detection에 필요한 CLIP backbone을 불러옵니다.
# base.load_models()는 head도 함께 불러오지만, 이 노트북에서는 예측을 하지 않고 CLIP backbone만 사용합니다.
clip_model, clip_preprocess, _, _ = base.load_models()

summary_rows = []

for download_index, (item_id, url) in enumerate(EXTRACTION_ITEMS, start=1):
    print("=" * 80)
    print(f"[{item_id}] extracting frames:", url)
    summary_rows.append(
        extract_scene_frames_only(
            item_id,
            url,
            download_index,
            clip_model,
            clip_preprocess,
        )
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(
    OUTPUT_ROOT / "clip_scene_frame_extraction_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

summary_df


c:\Users\eunpa\anaconda3\envs\sy\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


[0131] extracting frames: https://www.instagram.com/reel/DYjePvvTeSB/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA==
[Instagram] Extracting URL: https://www.instagram.com/reel/DYjePvvTeSB/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA==
[Instagram] DYjePvvTeSB: Setting up session
[Instagram] DYjePvvTeSB: Downloading JSON metadata
[info] DYjePvvTeSB: Downloading 1 format(s): 1
Deleting existing file C:\Temp\shot-classification\videos_new_links\new_001_DYjePvvTeSB.mp4
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_001_DYjePvvTeSB.mp4
[download] 100% of    1.42MiB in 00:00:01 at 1.00MiB/s   


[0132] extracting frames: https://www.instagram.com/reel/DV7sM11j9mk/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA==
[Instagram] Extracting URL: https://www.instagram.com/reel/DV7sM11j9mk/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA==
[Instagram] DV7sM11j9mk: Setting up session
[Instagram] DV7sM11j9mk: Downloading JSON metadata
[info] DV7sM11j9mk: Downloading 1 format(s): 6
Deleting existing file C:\Temp\shot-classification\videos_new_links\new_002_DV7sM11j9mk.mp4
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_002_DV7sM11j9mk.mp4
[download] 100% of    1.26MiB in 00:00:02 at 509.20KiB/s 


[0133] extracting frames: https://www.instagram.com/reel/DYvmCaHpVlW/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA==
[Instagram] Extracting URL: https://www.instagram.com/reel/DYvmCaHpVlW/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA==
[Instagram] DYvmCaHpVlW: Setting up session
[Instagram] DYvmCaHpVlW: Downloading JSON metadata
[info] DYvmCaHpVlW: Downloading 1 format(s): 3
Deleting existing file C:\Temp\shot-classification\videos_new_links\new_003_DYvmCaHpVlW.mp4
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_003_DYvmCaHpVlW.mp4
[download] 100% of    8.58MiB in 00:00:07 at 1.09MiB/s   


[0134] extracting frames: https://www.instagram.com/reel/DYcNmJLxAeM/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA==
[Instagram] Extracting URL: https://www.instagram.com/reel/DYcNmJLxAeM/?utm_source=ig_web_copy_link&igsh=MzRlODBiNWFlZA==
[Instagram] DYcNmJLxAeM: Setting up session
[Instagram] DYcNmJLxAeM: Downloading JSON metadata
[info] DYcNmJLxAeM: Downloading 1 format(s): 7
Deleting existing file C:\Temp\shot-classification\videos_new_links\new_004_DYcNmJLxAeM.mp4
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_004_DYcNmJLxAeM.mp4
[download] 100% of   10.79MiB in 00:00:07 at 1.41MiB/s   


...

[0193] extracting frames: https://www.instagram.com/reel/DYossBrSaf-/?igsh=eXk3amg4eGl3NnVx
[Instagram] Extracting URL: https://www.instagram.com/reel/DYossBrSaf-/?igsh=eXk3amg4eGl3NnVx
[Instagram] DYossBrSaf-: Setting up session
[Instagram] DYossBrSaf-: Downloading JSON metadata
[info] DYossBrSaf-: Downloading 1 format(s): 1
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_063_DYossBrSaf-.mp4
[download] 100% of    8.54MiB in 00:00:01 at 5.00MiB/s   


[0194] extracting frames: https://www.instagram.com/reel/DYt86KEStVK/?igsh=dTgzYjd0M2k0MWg1
[Instagram] Extracting URL: https://www.instagram.com/reel/DYt86KEStVK/?igsh=dTgzYjd0M2k0MWg1
[Instagram] DYt86KEStVK: Setting up session
[Instagram] DYt86KEStVK: Downloading JSON metadata
[info] DYt86KEStVK: Downloading 1 format(s): 9
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_064_DYt86KEStVK.mp4
[download] 100% of    3.78MiB in 00:00:00 at 7.13MiB/s   


[0195] extracting frames: https://www.instagram.com/reel/DW_UYoBviXv/?igsh=Nng5YzA2Z2l0YTht
[Instagram] Extracting URL: https://www.instagram.com/reel/DW_UYoBviXv/?igsh=Nng5YzA2Z2l0YTht
[Instagram] DW_UYoBviXv: Setting up session
[Instagram] DW_UYoBviXv: Downloading JSON metadata
[info] DW_UYoBviXv: Downloading 1 format(s): 1
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_065_DW_UYoBviXv.mp4
[download] 100% of    7.57MiB in 00:00:01 at 5.98MiB/s   


[0196] extracting frames: https://www.instagram.com/reel/DXgtgqJkSSK/?igsh=MWNwcW5ra3dlZ3phYg==
[Instagram] Extracting URL: https://www.instagram.com/reel/DXgtgqJkSSK/?igsh=MWNwcW5ra3dlZ3phYg==
[Instagram] DXgtgqJkSSK: Setting up session
[Instagram] DXgtgqJkSSK: Downloading JSON metadata
[info] DXgtgqJkSSK: Downloading 1 format(s): 8
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_066_DXgtgqJkSSK.mp4
[download] 100% of   11.67MiB in 00:00:01 at 11.62MiB/s  


,item_id,url,shortcode,video_path,output_dir,frames_dir,scene_count,duration_sec,threshold,num_extracted_frames
0,0131,https://www.instagram.com/reel/DYjePvvTeSB/?ut...,DYjePvvTeSB,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,C:\Temp\shot-classification\outputs_clip_frame...,4,6.600000,0.067046,4
1,0132,https://www.instagram.com/reel/DV7sM11j9mk/?ut...,DV7sM11j9mk,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,C:\Temp\shot-classification\outputs_clip_frame...,3,14.100000,0.055000,3
2,0133,https://www.instagram.com/reel/DYvmCaHpVlW/?ut...,DYvmCaHpVlW,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,C:\Temp\shot-classification\outputs_clip_frame...,8,19.800000,0.101666,8
3,0134,https://www.instagram.com/reel/DYcNmJLxAeM/?ut...,DYcNmJLxAeM,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,C:\Temp\shot-classification\outputs_clip_frame...,8,22.633333,0.071250,8
4,0135,https://www.instagram.com/reel/DWlU1QYEvCY/?ut...,DWlU1QYEvCY,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,C:\Temp\shot-classification\outputs_clip_frame...,4,18.200000,0.055000,4
...,...,...,...,...,...,...,...,...,...,...
61,0192,https://www.instagram.com/reel/DYB-OYRyLaO/?ig...,DYB-OYRyLaO,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,C:\Temp\shot-classification\outputs_clip_frame...,12,27.133333,0.168764,12
62,0193,https://www.instagram.com/reel/DYossBrSaf-/?ig...,DYossBrSaf-,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,C:\Temp\shot-classification\outputs_clip_frame...,19,47.066667,0.275104,19
63,0194,https://www.instagram.com/reel/DYt86KEStVK/?ig...,DYt86KEStVK,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,C:\Temp\shot-classification\outputs_clip_frame...,6,14.933333,0.139081,6
64,0195,https://www.instagram.com/reel/DW_UYoBviXv/?ig...,DW_UYoBviXv,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_clip_frame...,C:\Temp\shot-classification\outputs_clip_frame...,12,37.333333,0.126888,12


## 결과 확인 방법

실행 후 아래 구조로 결과가 생성됩니다.

```text
outputs_clip_frame_extraction/
  clip_scene_frame_extraction_summary.csv
  0131_SHORTCODE/
    scene_metadata.csv
    clip_distance_profile.csv
    scene_frames/
      0131_cut_001.jpg
      0131_cut_002.jpg
      0131_cut_003.jpg
```

`scene_frames/` 안의 이미지들을 확인한 뒤, 라벨 기준에 맞는 이미지만 `labeled_dataset/`의 적절한 폴더로 복사해서 사용하면 됩니다.
